# 🔥 Firecrawl — 让 AI 学会上网

> Web scraping + Search + Browser Interaction API，把任何网页变成 LLM-ready 的数据

## 今天你会学会：

| 知识点 | 说明 |
|:---|:---|
| 🔥 Firecrawl 是什么 | 一个让程序"上网"的 API |
| 📋 核心功能 | 搜索、抓取、交互、爬取、地图 |
| 🔐 免 key vs 有 key | 两种使用方式 |
| 💻 动手实战 | 爬网页、搜资料、做项目 |

---
# 1. Firecrawl 是什么？

**一句话：** 给 AI/程序用的"浏览器"。

你写 Python 代码让 Firecrawl 去**搜 Google、打开网页、提取内容、甚至点击按钮**，它把结果变成干净的 Markdown 或 JSON 返回给你。

### 为什么需要它？

- 普通 `requests.get(url)` — 遇到 JavaScript 网页就跪了
- 🎲 用 Selenium/Playwright — 配置麻烦，要装浏览器驱动
- 🔥 Firecrawl — 一个 API 搞定，JS 页面、反爬、登录都不怕

### 谁在用？

- AI Agent（Claude Code、OpenCode 等）用来上网查资料
- 开发者用来爬数据、做监控、建知识库
- 147k GitHub stars，Y Combinator 投资

### 核心架构

```
你的代码 (Python/JS)  -->  Firecrawl API  -->  目标网站
                           ↑ 自动处理：JS渲染、反爬、代理、
                           ↑ 速率限制、robots.txt 遵守
你收到 Markdown/JSON  <--  Firecrawl API  <--  数据返回
```

Firecrawl 就像你请了一个"专业爬虫司机"，你告诉它去哪，它负责开车、绕路、加油——你只管收结果。

---
# 2. Firecrawl 有什么功能？

| 功能 | 一句话 | 类比 |
|:---|:---|:---|
| 🔍 **Search** | 搜关键词，返回网页内容 | 让 Firecrawl 帮你 Google 并读完结果 |
| 📥 **Scrape** | 抓取单个网页内容 | 让 Firecrawl 打开一个网页，复制给你 |
| 🤖 **Interact** | 在网页上点击、输入、滚动 | 让 Firecrawl 替你操作网页 |
| 👷 **Agent** | 用一句话描述需求，AI 自动爬 | 你只管说"帮我找..." |
| 📚 **Crawl** | 爬取整个网站所有页面 | 让 Firecrawl 读完一本书（所有页面） |
| 🗺 **Map** | 发现网站上所有 URL | 让 Firecrawl 告诉你这本书有哪些章节 |
| 📦 **Batch Scrape** | 同时抓取多个 URL | 一次撕下多页内容 |

> 💡 **核心定位**：Firecrawl 不是普通的爬虫，它是**面向 AI 的"网络数据接口"**。输出的是 Markdown/JSON，AI 直接能读。

---
# 3. 准备工作

### 方式 A：免 key 直接跑（推荐先试）

Firecrawl 开放了**免 key 入口**，让你零摩擦体验。直接调 API，不传 `Authorization` header 即可。

> ⚠️ 限制：速率有限、只有 search/scrape/interact 三个功能。
> 但用来学**完全够了**。

### 方式 B：注册拿 key（正式使用）

去 [firecrawl.dev/signin](https://www.firecrawl.dev/signin?view=signup) 注册，免费送 1000 次/月。

拿到 key 后：

```python
headers = {
    "Authorization": "Bearer fc-YOUR_API_KEY"
}
```

本节课**两种方式都会演示**，先用免 key 跑通，再教你用 key。

### 安装依赖

只需要 `requests`，我们已经装过了：

In [ ]:
# 检查 requests 是否安装
try:
    import requests
    print("✅ requests 已安装，版本:", requests.__version__)
except ImportError:
    print("❌ 请运行: pip install requests")

In [ ]:
# Firecrawl API 的基础地址
BASE_URL = "https://api.firecrawl.dev/v2"

# 免 key 模式：不传 Authorization header
# 有 key 模式：传 headers = {"Authorization": "Bearer fc-xxx"}

# 我们先定义一个工具函数，方便后面调用
import requests, json

def firecrawl(endpoint, data, api_key=None):
    """调用 Firecrawl API
    endpoint: /search, /scrape 等
    data: 请求体字典
    api_key: 传了就带 key，不传就免 key
    """
    url = BASE_URL + endpoint
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    
    r = requests.post(url, headers=headers, json=data)
    print(f"状态码: {r.status_code}")
    return r.json()

print("✅ 工具函数就绪！")
print("用法: firecrawl('/endpoint', {...}, api_key='...')")

---
# 4. Scrape — 抓取单个网页

**Scrape** 是 Firecrawl 最核心的功能：给它一个 URL，它返回网页的**干净内容**（Markdown 格式）。

**对比**：
- 💥 `requests.get()` → 拿到 HTML，全是标签，自己解析
- 🔥 Firecrawl Scrape → 直接拿到 Markdown，干净、AI 可读

**支持的格式（`formats` 参数）**：
- `"markdown"` — 干净的文字（默认，最常用）
- `"html"` — 原始 HTML
- `"screenshot"` — 网页截图（Base64）
- `"json"` — 结构化数据

> example.com 是由互联网号码分配局（IANA）管理的保留域名。它不能被注册使用，主要用于在各类技术文档、书籍或示例中提供一个安全的占位符，避免人们在演示时误用真实的网站。当你看到这个网址时，通常意味着这是一个标准示例。

In [ ]:
# 实战 1：抓取一个网页（免 key）
result = firecrawl("/scrape", {
    "url": "https://example.com"
})

# 看看返回的结构
print(json.dumps(result, indent=2, ensure_ascii=False)[:800])

In [ ]:
# 提取 Markdown 内容
result = firecrawl("/scrape", {
    "url": "https://example.com"
})

if "data" in result:
    content = result["data"]["markdown"]
    print("=" * 40)
    print(content[:600])
    print("=" * 40)
    print(f"\n总长度: {len(content)} 字符")
else:
    print("出错了:", result)

In [ ]:
# 实战 2：抓取不同网站看看效果

urls = [
    "https://www.python.org",
    "https://github.com/trending/python",
]

for url in urls:
    print(f"\n{'='*50}")
    print(f"\U0001f4cc 正在抓取: {url}")
    result = firecrawl("/scrape", {"url": url})
    if "data" in result:
        md_content = result["data"]["markdown"]
        print(f"内容长度: {len(md_content)} 字符")
        print(f"前 200 字:\n{md_content[:200]}")
    else:
        print(f"失败: {result.get('error', '未知错误')}")

### Scrape 进阶参数

| 参数 | 作用 | 示例 |
|:---|:---|:---|
| `formats` | 返回格式列表 | `["markdown", "screenshot"]` |
| `onlyMainContent` | 只取正文，去掉导航/广告 | `true` |
| `waitFor` | 等待页面加载(ms) | `2000` |
| `timeout` | 超时时间(ms) | `30000` |


In [ ]:
# 实战 3：爬取带 JavaScript 的页面（比如新闻站）
result = firecrawl("/scrape", {
    "url": "https://news.ycombinator.com",
    "formats": ["markdown"],
    "onlyMainContent": True
})

if "data" in result:
    print(result["data"]["markdown"][:1000])
else:
    print("结果:", json.dumps(result, indent=2, ensure_ascii=False)[:500])

---
# 5. Search — 搜索网页

Search 相当于"Google + 读完所有结果"：搜关键词，Firecrawl 不仅给你链接，还直接把每个结果的内容一起返回。

**适用场景**：
- AI 需要查资料
- 竞品分析
- 找教程/文档
- 实时信息获取


In [ ]:
# 实战 4：搜索 Python 教程（免 key）
result = firecrawl("/search", {
    "query": "Python 入门教程 2026",
    "limit": 3
})

# Firecrawl v2 search 返回结构：用 .get() 安全取值
if "data" in result:
    data = result["data"]
    # v2 的搜索结果可能在 "web" 里，也可能直接就是列表
    results = data.get("results") or data.get("web") or data.get("items", [])
    if isinstance(results, list) and len(results) > 0:
        for i, item in enumerate(results):
            print(f"\n{'='*50}")
            print(f"结果 {i+1}")
            print(f"标题: {item.get('title', '无')}")
            print(f"链接: {item.get('url', '无')}")
            print(f"内容: {str(item.get('markdown', item.get('content', '')))[:200]}...")
    else:
        print("API 返回结构：")
        print(json.dumps(data, indent=2, ensure_ascii=False)[:500])
else:
    print("结果:", json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# 实战 5：搜索 AI 新闻
result = firecrawl("/search", {
    "query": "AI agent 2026 open source",
    "limit": 3
})

if "data" in result:
    data = result["data"]
    results = data.get("results") or data.get("web") or data.get("items", [])
    if isinstance(results, list) and len(results) > 0:
        for i, item in enumerate(results):
            print(f"\n--- 结果 {i+1} ---")
            print(f"标题: {item.get('title', '无')}")
            print(f"链接: {item.get('url', '无')}")
            content = str(item.get('markdown', item.get('content', '')))
            print(f"内容: {content[:200]}...")
    else:
        print("API 返回结构：")
        print(json.dumps(data, indent=2, ensure_ascii=False)[:500])
else:
    print("结果:", json.dumps(result, indent=2, ensure_ascii=False))

### Search vs Google 搜索

| | Google | Firecrawl Search |
|:---|:---|:---|
| 返回 | 10 个蓝色链接 | 链接 + 每个页面的完整内容 |
| 格式 | HTML 页面 | Markdown/JSON |
| 给 AI 用 | ❌ 需要再爬一次 | ✅ 一次搞定 |
| 个性化 | 追踪你 | 干净结果 |

---
# 6. Interact — 和网页交互

**Interact** 让 Firecrawl 像真人一样操作网页：点击、输入文字、滚动、等待。

**工作流程**：
```
Step 1: Scrape 页面 → 拿到 scrape_id
Step 2: Interact 操作 → 告诉它做什么
Step 3: 拿到操作后的结果
```

**支持的动作**（通过 `prompt` 参数描述）：
- 👆 "Click the login button"
- ⌨️ "Search for mechanical keyboard"
- 📜 "Scroll down"
- ⏳ "Wait 3 seconds"


In [ ]:
# 实战 6：用 key 搜索（更多额度）
MY_API_KEY = "fc-YOUR_API_KEY_HERE"

result = firecrawl("/search", {
    "query": "Python tutorial 2026",
    "limit": 5
}, api_key=MY_API_KEY)

if result.get("success") is False:
    print("❌ API 返回错误:", result.get("error", "未知错误"))
elif "data" in result:
    data = result["data"]
    results = data.get("results") or data.get("web") or data.get("items", [])
    if isinstance(results, list) and len(results) > 0:
        print(f"✅ 找到 {len(results)} 个结果:\n")
        for i, item in enumerate(results):
            print(f"{i+1}. {item.get('title', '无标题')}")
            print(f"   {item.get('url', '')[:80]}")
            print()
    else:
        print("API 返回结构：")
        print(json.dumps(data, indent=2, ensure_ascii=False)[:500])
else:
    print("原始返回:", json.dumps(result, indent=2, ensure_ascii=False)[:500])

> ⚠️ **注意**：Interact 的完整功能建议在有 API key 的环境下使用。
> 免 key 模式主要体验 search 和 scrape。

---
# 7. Crawl — 爬取整个网站

Crawl 能自动爬完一个网站的所有页面（或者指定数量）。

**适合场景**：
- 爬文档网站
- 备份博客
- 构建知识库

> ⚠️ Crawl 需要 API key，免 key 不可用。


In [ ]:
# Crawl 示例（需要 API key）
# 如果没 key，这段会报错，先看看代码结构

api_key = input("输入你的 Firecrawl API key（没有就直接回车跳过）: ")

if api_key:
    result = firecrawl("/crawl", {
        "url": "https://docs.firecrawl.dev",
        "limit": 5,
        "scrapeOptions": {
            "formats": ["markdown"]
        }
    }, api_key=api_key)
    print(json.dumps(result, indent=2, ensure_ascii=False)[:500])
else:
    print("\U0001f4a1 跳过 Crawl（需要 API key）")
    print("注册 https://firecrawl.dev 免费拿 1000 次/月")

### Map — 发现网站结构

Map 不抓内容，只返回一个网站有哪些 URL。相当于"网站的目录"。

In [ ]:
# Map 示例
api_key = input("输入你的 Firecrawl API key（没有就直接回车跳过）: ")

if api_key:
    result = firecrawl("/map", {
        "url": "https://www.python.org"
    }, api_key=api_key)

    if "data" in result:
        links = result["data"]["links"]
        print(f"\U0001f5fa 发现 {len(links)} 个链接\n")
        for link in links[:10]:
            print(f"  \U0001f517 {link.get('url', link) if isinstance(link, dict) else link}")
        if len(links) > 10:
            print(f"  ... 还有 {len(links) - 10} 个")
    else:
        print("结果:", json.dumps(result, indent=2, ensure_ascii=False)[:300])
else:
    print("💡 跳过 Map（需要 API key）")
    print("注册 https://firecrawl.dev 免费拿 1000 次/月")

---
# 8. 有 Key 模式 vs 免 Key 模式

## 对比总结

| 维度 | 🚫 免 Key | 🔑 有 Key（免费版） |
|:---|:---|:---|
| Search | ✅ | ✅ |
| Scrape | ✅ | ✅ |
| Interact | ✅（有限） | ✅（完整） |
| Crawl | ❌ | ✅ |
| Map | ✅ | ✅ |
| Batch Scrape | ❌ | ✅ |
| Agent | ❌ | ✅ |
| 速率 | 很慢 | 2 并发 |
| 每月配额 | 无 | 1000 次免费 |

## 什么时候用 key？

- 你要爬完整网站（Crawl）
- 你要批量抓取（Batch Scrape）
- 你要用 Agent 功能
- 免 key 速率不够用了

## 怎么拿 key？

1. 打开 https://firecrawl.dev/signin
2. 注册账号
3. 去 Dashboard 复制 API Key
4. 每月 1000 次免费，不需要绑信用卡


In [ ]:
# 实战 7：有 Key 模式完整示例
# 填入你的 key 试试
MY_API_KEY = input("粘贴你的 Firecrawl API Key（没有就回车）: ")

if MY_API_KEY:
    # 1. Scrape
    print("\n📥 1. 抓取网页")
    r = firecrawl("/scrape", {
        "url": "https://www.python.org",
        "formats": ["markdown"],
        "onlyMainContent": True
    }, api_key=MY_API_KEY)
    print(r["data"]["markdown"][:200] if "data" in r else r)
    
    # 2. Search
    print("\n🔍 2. 搜索")
    r = firecrawl("/search", {
        "query": "Python web scraping",
        "limit": 2
    }, api_key=MY_API_KEY)
    if "data" in r:
        data = r["data"]
        results = data.get("results") or data.get("web") or data.get("items", [])
        for item in results:
            print(f"  - {item.get('title', '无标题')}")
else:
    print("💡 跳过，用上面的免 key 示例也一样")

---
# 🎮 综合实战：Web 信息收集器

做一个 Python 程序，输入关键词，自动搜索并抓取相关内容。

In [ ]:
# 综合实战：网页收集器
MY_API_KEY = "fc-YOUR_API_KEY_HERE"

def web_collector(query, max_results=3, api_key=None):
    """搜索 + 抓取 + 保存"""
    print(f"🔍 搜索: {query}")
    search_result = firecrawl("/search", {"query": query, "limit": max_results}, api_key=api_key)

    if "data" not in search_result:
        print("搜索失败")
        return

    data = search_result["data"]
    results = data.get("results") or data.get("web") or data.get("items", [])
    if not isinstance(results, list) or len(results) == 0:
        print("没有找到结果，API 返回结构：")
        print(json.dumps(data, indent=2, ensure_ascii=False)[:300])
        return

    print(f"找到 {len(results)} 个结果\n")

    # Step 2: 保存到文件
    filename = f"search_results_{query.replace(' ', '_')}.md"
    with open(filename, "w", encoding="utf-8") as f:
        for i, item in enumerate(results):
            title = item.get("title", "无标题")
            url = item.get("url", "")
            content = str(item.get("markdown", item.get("content", "")))
            f.write(f"## {i+1}. {title}\n\n")
            f.write(f"🔗 {url}\n\n")
            f.write(f"{content[:300]}...\n\n---\n\n")

    print(f"✅ 已保存到 {filename}")
    print(f"文件大小: {len(open(filename, 'r', encoding='utf-8').read())} 字符")

# 运行试试
web_collector("Python 教程 2026", max_results=2)

In [ ]:
# 看一下保存的文件内容
try:
    with open("web_collect_Python_教程_2.md", "r", encoding="utf-8") as f:
        print(f.read()[:800])
except:
    print("文件未找到，先运行上面的 cell")

---
# 📝 For Practice

## 练习 1：对比 requests vs Firecrawl

用 `requests.get()` 和 Firecrawl 分别抓同一个 URL，对比返回的内容：

In [ ]:
# 练习 1 模板
import requests as req

url = "https://www.python.org"

# requests 方式
print("=== requests.get() 返回 ===")
r = req.get(url)
print(f"类型: {type(r.text)}")
print(f"长度: {len(r.text)} 字符")
print(f"前 200 字: {r.text[:200]}")
print()

# Firecrawl 方式
print("=== Firecrawl Scrape 返回 ===")
result = firecrawl("/scrape", {"url": url})
if "data" in result:
    print(f"类型: markdown")
    print(f"长度: {len(result['data']['markdown'])} 字符")
    print(f"前 200 字: {result['data']['markdown'][:200]}")

# 💡 对比：哪个更干净？哪个 AI 更容易读懂？

## 练习 2：做一个"网页摘要器"

输入 URL，抓取内容，打印前 300 字。

In [ ]:
# 练习 2 模板
def page_summarizer(url):
    """输入 URL，返回网页摘要"""
    result = firecrawl("/scrape", {"url": url})
    if "data" in result:
        md = result["data"]["markdown"]
        print(f"📌 {url}")
        print(f"=" * 40)
        print(md[:300])
        print(f"=" * 40)
        print(f"总字数: {len(md)}")
    else:
        print(f"抓取失败: {result}")

# 测试
page_summarizer("https://en.wikipedia.org/wiki/Python_(programming_language)")

## 练习 3：搜索 + 筛选

搜一个你感兴趣的话题，把结果中标题包含某个关键词的打印出来。

In [ ]:
# 练习 3 模板
MY_API_KEY = "fc-YOUR_API_KEY_HERE"

def search_and_filter(query, keyword, limit=5):
    """搜索并过滤包含指定关键词的结果"""
    print(f"🔍 搜索: {query}，过滤关键词: {keyword}")
    result = firecrawl("/search", {"query": query, "limit": limit}, api_key=MY_API_KEY)

    if "data" not in result:
        print("搜索失败")
        return

    data = result["data"]
    results = data.get("results") or data.get("web") or data.get("items", [])
    if not isinstance(results, list):
        print("API 返回结构：")
        print(json.dumps(data, indent=2, ensure_ascii=False)[:300])
        return

    filtered = []
    for item in results:
        title = item.get("title", "")
        if keyword.lower() in title.lower():
            filtered.append(item)

    if filtered:
        print(f"找到 {len(filtered)} 条包含「{keyword}」的结果:\n")
        for item in filtered:
            print(f"  📌 {item.get('title', '无标题')}")
            print(f"     {item.get('url', '')}")
            print()
    else:
        print(f"没有找到包含「{keyword}」的结果")

# 试试
search_and_filter("AI tools 2026", "GPT", limit=5)

## 练习 4：挑战 — 多页爬取

选一个你喜欢的博客，用 Firecrawl 抓取首页，提取所有文章链接。

In [ ]:
# 练习 4 模板 — 挑战
# 💡 提示：先用 Map 发现 URL，再用 Scrape 抓每个页面

def crawl_blog(blog_url, api_key=None):
    """爬取博客首页，提取文章"""
    print(f"🗺 正在分析: {blog_url}")
    
    # Step 1: Scrape 首页
    result = firecrawl("/scrape", {"url": blog_url}, api_key)
    if "data" not in result:
        print("抓取失败")
        return
    
    md = result["data"]["markdown"]
    print(f"首页内容长度: {len(md)} 字符")
    print(f"\n首页前 500 字:\n{md[:500]}")

# 测试
crawl_blog("https://github.blog")

---
# 💡 总结

## Firecrawl 核心一句话

> **Firecrawl = 给 AI 用的"上网技能"** — 搜、看、点、读，全在一个 API 里。

## 今天学到的

| # | 内容 | 状态 |
|:---:|:---|:---:|
| 1 | Firecrawl 是什么、为什么需要它 | ✅ |
| 2 | Scrape — 抓单个网页 | ✅ |
| 3 | Search — 搜索 + 获取内容 | ✅ |
| 4 | Interact — 操作网页（点击/输入） | ✅ |
| 5 | Crawl — 爬整个网站 | ✅（需 key） |
| 6 | Map — 发现网站结构 | ✅ |
| 7 | 免 key vs 有 key | ✅ |
| 8 | 综合实战：信息收集器 | ✅ |

## 继续探索

| 方向 | 链接 |
|:---|:---|
| 📖 官方文档 | https://docs.firecrawl.dev |
| 🚀 GitHub | https://github.com/firecrawl/firecrawl |
| 🖥 Playground | https://firecrawl.dev/playground |
| 🤝 社区 Discord | https://discord.gg/firecrawl |


---
# 📝 Firecrawl CLI 速查表

> 所有命令一键运行，不用写 Python 代码。有 `-k fc-xxx` 的就是需要 API key，用你的 key 替换 `xxx`。

## 🛠️ 安装 CLI

**全局安装（推荐，一次装完到处用）**：

```bash
npm install -g firecrawl
```

**或者不装直接用 npx**（每次自动下载最新版）：

```bash
npx firecrawl@latest <命令>
```

> ✅ 下面所有命令都假设你已经全局安装了。如果你没装，把 `firecrawl` 替换成 `npx -y firecrawl@latest`。

## 📥 scrape — 抓取单个网页

**作用**：给一个 URL，返回网页的干净内容（Markdown）。

**免 key ✅**

```bash
firecrawl scrape https://...
```

**有 key**

```bash
firecrawl -k fc-YOUR_KEY scrape https://...
```

In [ ]:
# 📥 scrape 示例 — 免 key
!firecrawl scrape https://www.python.org

## 🔍 search — 搜索网页

**作用**：搜关键词，直接返回每个结果的标题 + 链接 + 内容片段。

**免 key ✅**

```bash
firecrawl search "关键词"
```

**有 key**

```bash
firecrawl -k fc-YOUR_KEY search "关键词"
```

In [ ]:
# 🔍 search 示例 — 免 key
!firecrawl search "Python tutorial"

## 🤖 interact — 操作网页

**作用**：让 Firecrawl 在网页上点击、输入、滚动，像一个真人一样操作。

**使用流程**：先 scrape 拿到页面 ID，再对它做操作。

**免 key ✅**（有限制）

```bash
firecrawl scrape https://...
# 复制返回的 Scrape ID，或直接用（自动用上次 ID）
firecrawl interact "点第一个链接"
# 或者指定 ID:
firecrawl interact -s SCRAPE_ID "点第一个链接"
```

**有 key**

```bash
firecrawl -k fc-YOUR_KEY scrape https://...
firecrawl -k fc-YOUR_KEY interact "点第一个链接"
```

In [ ]:
# 🤖 interact 示例 — 免 key（先抓取页面）
!firecrawl scrape https://news.ycombinator.com

> 拿到上面的 Scrape ID，替换下面命令中的 `SCRAPE_ID` 再运行。


In [ ]:
# 用 Scrape ID 操作页面
# firecrawl interact <scrape-id> "点第一个链接"
# !firecrawl interact -s <scrape-id> "点第一个链接"
# 或者直接用（自动用上次 scrape 的 ID）: 
!firecrawl interact "点第一个链接"

## 🗺 map — 发现网站结构

**作用**：不抓内容，只返回一个网站有哪些页面。相当于网站的"目录"。

**需要 API key ❌**

```bash
firecrawl -k fc-YOUR_KEY map https://...
```

> 💡 免费注册 https://firecrawl.dev 每月送 1000 次

In [ ]:
# 🗺 map 示例 — 需要 key
# 把 fc-YOUR_KEY 换成你的真实 key
!firecrawl -k fc-YOUR_KEY map https://www.python.org

## 📚 crawl — 爬取整个网站

**作用**：自动爬完一个网站的所有页面（或指定数量）。

**需要 API key ❌**

```bash
firecrawl -k fc-YOUR_KEY crawl https://... --limit 10 --wait
```

`--limit 10` = 只爬 10 页，不设会爬完整个站。
`--wait` = 等爬完再返回结果，不加只返回 job ID。

In [ ]:
# 📚 crawl 示例 — 需要 key（--wait 等结果）
!firecrawl -k fc-YOUR_KEY crawl https://docs.firecrawl.dev --limit 5 --wait

## 👷 agent — AI 自动爬取

**作用**：用一句话描述需求，AI 自动搜索、导航、提取数据。不用指定 URL。

**需要 API key ❌**

```bash
firecrawl -k fc-YOUR_KEY agent "帮我找到 Python 官网的所有教程链接" --wait
```

`--wait` = 等 Agent 跑完再返回结果，不加只返回 job ID。

In [ ]:
# 👷 agent 示例 — 需要 key（--wait 等结果）
# Agent 是异步的，不加 --wait 只返回 job ID
!firecrawl -k fc-YOUR_KEY agent "列出 Python 官网的免费教程" --wait

---
## 💡 总结

| 功能 | 免 key | 命令 |
|:---|:---:|:---|
| scrape | ✅ | `firecrawl scrape https://...` |
| search | ✅ | `firecrawl search "关键词"` |
| interact | ✅ | `firecrawl interact "操作"` |
| map | ❌ | `firecrawl -k fc-xxx map https://...` |
| crawl | ❌ | `firecrawl -k fc-xxx crawl https://... --limit 10 --wait` |
| agent | ❌ | `firecrawl -k fc-xxx agent "需求" --wait` |

> - 有 key 就是把 `-k fc-YOUR_KEY` 加在命令里
> - `--wait` = 等结果（agent 和 crawl 是异步的，不加只返回 job ID）
> - scrape 和 search 是同步的，不用 `--wait`
> - 如果没全局安装，把 `firecrawl` 替换成 `npx -y firecrawl@latest`


---
## 🆘 查看完整命令列表

任何时候忘了命令，直接：

```bash
firecrawl --help
```

### 全局选项

| 选项 | 作用 |
|:---|:---|
| `-k, --api-key <key>` | 指定 API key（或设 `FIRECRAWL_API_KEY` 环境变量） |
| `--api-url <url>` | 指定 API 地址（或设 `FIRECRAWL_API_URL` 环境变量） |
| `--status` | 查看版本、登录状态、并发数、剩余积分 |
| `-V, --version` | 查看版本号 |
| `-h, --help` | 查看帮助 |

### 🕸️ 核心爬取命令

| 命令 | 用途 |
|:---|:---|
| `scrape [urls...]` | 抓取单个或多个 URL，返回 Markdown |
| `search <query>` | 搜索网页，返回标题+链接+内容 |
| `crawl [url]` | 爬取整个网站（需 key） |
| `map [url]` | 发现网站所有 URL（需 key） |
| `agent <prompt>` | AI 自动完成爬取任务（需 key） |
| `interact [args...]` | 操作网页：点击、输入、滚动 |

### 📄 其他数据处理

| 命令 | 用途 |
|:---|:---|
| `parse <file>` | 解析本地文件（HTML、PDF、DOCX、XLSX 等）→ Markdown |
| `research` | 搜索 arXiv 论文和 GitHub 历史 |
| `monitor` | 定时监控网页内容变化 |
| `feedback <endpoint> <jobId>` | 提交任务反馈 |
| `search-feedback <searchId>` | 提交搜索结果反馈（退 1 积分） |

### ⚙️ 配置与管理

| 命令 | 用途 |
|:---|:---|
| `login` | 登录 Firecrawl |
| `logout` | 退出登录 |
| `config` | 配置 Firecrawl |
| `view-config` | 查看当前配置和登录状态 |
| `env` | 把 key 写入本地 `.env` 文件 |
| `credit-usage` | 查看团队积分使用情况 |
| `doctor [job-id]` | 诊断环境或排查某个任务 |

### 🧩 集成与启动

| 命令 | 用途 |
|:---|:---|
| `init [template]` | 一键初始化：安装 CLI、登录、集成 |
| `setup [subcommand]` | 单独配置某个集成（skills / MCP 等） |
| `make <target>` | 把 Firecrawl 设为默认 provider |
| `launch [agent]` | 给 AI agent 配好 MCP 然后启动它 |
| `version` | 显示版本信息 |
| `help [command]` | 查看某个命令的详细帮助 |
| `experimental|x` | 实验性功能（下载等） |

> 💡 任何命令加 `--help` 看详细参数，如 `firecrawl crawl --help`


In [ ]:
# 运行help
!firecrawl --help